In [4]:
import random
from collections import Counter

def diversity_reward(completions, **kwargs) -> list[float]:
    """Reward unique completions, penalize identical ones to prevent mode collapse.

    Uses random tiebreaking so identical completions get different rewards,
    creating gradient signal even when all outputs collapse to the same text.
    """
    texts = []
    for completion in completions:
        if isinstance(completion, list):
            text = completion[0].get('content', '') if completion else ''
        else:
            text = str(completion)
        texts.append(text.lower().strip())

    counts = Counter(texts)
    n = len(texts)

    # Track how many of each duplicate we've seen for tiebreaking
    seen_counts = {}

    rewards = []
    for text in texts:
        frequency = counts[text] / n

        # Base penalty for duplicates
        base_reward = 1.0 - (2.0 * frequency)

        # Add decreasing reward for each duplicate instance
        # First occurrence gets base, subsequent ones get progressively worse
        if text not in seen_counts:
            seen_counts[text] = 0
        seen_counts[text] += 1

        # Tiebreaker: penalize later duplicates more heavily
        duplicate_penalty = -1.0 * (seen_counts[text] - 1) / n

        # Small random noise to break exact ties in gradient computation
        noise = random.uniform(-0.01, 0.01)

        rewards.append(base_reward + duplicate_penalty + noise)

    return rewards


# Test with 5 identical completions
completions = [
    "left right right",
    "left right right",
    "left right right",
    "left right right",
    "left right right",
]

rewards = diversity_reward(completions)
print("Completions:", completions)
print("Rewards:", rewards)
print()
for i, (c, r) in enumerate(zip(completions, rewards)):
    print(f"  {i}: '{c}' -> reward = {r:.4f}")

Completions: ['left right right', 'left right right', 'left right right', 'left right right', 'left right right']
Rewards: [-1.0068342388297733, -1.0929493719235732, -1.1990936646849562, -1.3075933881955442, -1.3974569987834544]

  0: 'left right right' -> reward = -1.0068
  1: 'left right right' -> reward = -1.0929
  2: 'left right right' -> reward = -1.1991
  3: 'left right right' -> reward = -1.3076
  4: 'left right right' -> reward = -1.3975


In [5]:
import reasoning_gym
class Maze:
    def __init__(self, dataset_name="shortest_path", min_rows=3, max_rows=3, min_cols=3, max_cols=3, p_blocked=0.4, size=10, seed=None):
        """Initialize maze with dataset parameters"""
        self.min_rows = min_rows
        self.max_rows = max_rows
        self.min_cols = min_cols
        self.max_cols = max_cols
        self.p_blocked = p_blocked
        self.size = size
        self.dataset = reasoning_gym.create_dataset(
            dataset_name,
            min_rows=min_rows,
            max_rows=max_rows,
            min_cols=min_cols,
            max_cols=max_cols,
            p_blocked=p_blocked,
            size=size,
            seed=seed
        )

maze = Maze()

In [9]:
list(maze.dataset)[0]

{'question': 'Your task is to find the shortest path from the start to the destination point in a grid.\n\nThe grid is represented as a matrix with the following types of cells:\n- *: your starting point\n- #: your destination point\n- O: an open cell\n- X: a blocked cell\n\nTherefore, you need to find the shortest path from * to #, moving only through open cells.\n\nYou may only move in four directions: up, down, left, and right.\n\nIf there is no path from * to #, simply write "infeasible" (without quotes).\n\nYour output should be a sequence of directions that leads from * to #, e.g. right right down down up left\n\nNow, find the length of the shortest path from * to # in the following grid:\nO X O\n* # X\nO X X\n',
 'answer': 'right',
 'metadata': {'source_dataset': 'shortest_path',
  'source_index': 0,
  'matrix': [['O', 'X', 'O'], ['*', '#', 'X'], ['O', 'X', 'X']],
  'solution': ['right'],
  'difficulty': {'rows': (3, 3), 'cols': (3, 3)}}}

In [11]:
import ast
ast.literal_eval("{'source_dataset': 'shortest_path','source_index': 0,'matrix': [['X' 'O', 'O'], ['X', '#', 'X'], ['X', '*', 'X']],'solution': ['up'],'difficulty': {'rows': (3, 3), 'cols': (3, 3)}}")

{'source_dataset': 'shortest_path',
 'source_index': 0,
 'matrix': [['XO', 'O'], ['X', '#', 'X'], ['X', '*', 'X']],
 'solution': ['up'],
 'difficulty': {'rows': (3, 3), 'cols': (3, 3)}}

In [10]:
list(maze.dataset)[0]['metadata']

{'source_dataset': 'shortest_path',
 'source_index': 0,
 'matrix': [['X', 'O', 'O'], ['X', '#', 'X'], ['X', '*', 'X']],
 'solution': ['up'],
 'difficulty': {'rows': (3, 3), 'cols': (3, 3)}}

In [5]:
list(maze.dataset)[0]['metadata']['matrix']

[['X', 'O', 'O'], ['X', '#', 'X'], ['X', '*', 'X']]

['[',
 '[',
 "'",
 'X',
 "'",
 ',',
 ' ',
 "'",
 'O',
 "'",
 ',',
 ' ',
 "'",
 'O',
 "'",
 ']',
 ',',
 ' ',
 '[',
 "'",
 'X',
 "'",
 ',',
 ' ',
 "'",
 '#',
 "'",
 ',',
 ' ',
 "'",
 'X',
 "'",
 ']',
 ',',
 ' ',
 '[',
 "'",
 'X',
 "'",
 ',',
 ' ',
 "'",
 '*',
 "'",
 ',',
 ' ',
 "'",
 'X',
 "'",
 ']',
 ']']

In [5]:
import torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM,
DataCollatorForLanguageModeling, TrainingArguments, Trainer)

from peft import LoraConfig, get_peft_model
# Select model
model_name_path = "models/gemma_3.1_4B_instruct"

device = "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16
# Load tokenizer
tok = AutoTokenizer.from_pretrained(model_name_path, use_fast=True)
tok.pad_token = tok.eos_token


In [6]:
# Load model
model = AutoModelForCausalLM.from_pretrained(model_name_path, dtype=torch.float16)
model.to(device)

Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.79s/it]


Gemma3ForConditionalGeneration(
  (model): Gemma3Model(
    (vision_tower): SiglipVisionModel(
      (vision_model): SiglipVisionTransformer(
        (embeddings): SiglipVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
          (position_embedding): Embedding(4096, 1152)
        )
        (encoder): SiglipEncoder(
          (layers): ModuleList(
            (0-26): 27 x SiglipEncoderLayer(
              (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (self_attn): SiglipAttention(
                (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
              )
              (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwi